In [ ]:
import sys
import os

# Agrega la raíz del proyecto para poder importar desde src
sys.path.append(os.path.abspath('..'))

import pandas as pd
from src.utils import verificar_calidad

### 1: Importación de Librerías
Cargamos las librerías necesarias

In [ ]:
import pandas as pd
import numpy as np
import os

# Configuración para ver todas las columnas del DataFrame
pd.set_option('display.max_columns', None)

### 2: Carga del dataset principal (bank-additional.csv)
>>2.1. Verificamos si se usa como delimitador coma o punto y coma

In [ ]:
# Carga inicial de df_bank
df_bank = pd.read_csv('../data/raw/bank-additional.csv', sep=';')

# Ver el nombre exacto de todas las columnas de df_bank
print(df_bank.columns.tolist())

>> 2.2. Confirmando que "," es el delimitador comprobaremos con df_bank.head() si se lee correctamente

In [ ]:
# Ruta del archivo CSV
path_bank = os.path.join('..', 'data', 'raw', 'bank-additional.csv')

# Carga del CSV
df_bank = pd.read_csv(path_bank, sep=',')

print("Dimensiones de df_bank:", df_bank.shape)
df_bank.head()

### 3: Carga y unificación de las 3 hojas del Excel (customer-details.xlsx)
El documento del proyecto especifica que el archivo customer-details.xlsx consta de 3 hojas de trabajo diferentes con clientes agregados en diferentes años.  
Para cargarlas todas de forma limpia y eficiente en un solo DataFrame, leemos el diccionario completo de hojas con sheet_name=None y las concatenamos con pd.concat()

In [ ]:
# Ruta del archivo Excel
path_customer = os.path.join('..', 'data', 'raw', 'customer-details.xlsx')

# Leer todas las hojas del Excel
excel_sheets = pd.read_excel(path_customer, sheet_name=None)

# Mostrar los nombres de las hojas encontradas
print("Hojas encontradas en el Excel:", list(excel_sheets.keys()))

# Unir todas las hojas en un único DataFrame
df_customer = pd.concat(excel_sheets.values(), ignore_index=True)

print("Dimensiones de df_customer unificado:", df_customer.shape)
df_customer.head()

### 4: Verificación de duplicados antes de cruzar
Antes de hacer la unión de los datasets, comprobamos si existen IDs duplicados en df_customer o df_bank para evitar duplicación de registros no deseada

In [ ]:
print("IDs duplicados en df_customer:", df_customer['ID'].duplicated().sum())
print("IDs duplicados en df_bank:", df_bank['id_'].duplicated().sum())

### 5: Cruce (Merge) de los Datasets
5.1 Unimos ambos DataFrames relacionando la clave única "id_" de df_bank con "ID" de df_customer

In [ ]:
# Realizar el merge por las claves id_ e ID
df_merged = pd.merge(df_bank, df_customer, left_on='id_', right_on='ID', how='inner')

# Eliminar una de las columnas ID duplicadas tras el cruce
df_merged.drop(columns=['ID'], inplace=True)

print("Dimensiones del DataFrame combinado (df_merged):", df_merged.shape)
df_merged.head()

5.2 Eliminamos las columnas "Unnamed"

In [ ]:
# Buscar y eliminar cualquier columna que contenga 'Unnamed'
cols_unnamed = [col for col in df_merged.columns if 'unnamed' in col.lower()]
df_merged.drop(columns=cols_unnamed, inplace=True)

print(f"Columnas eliminadas: {cols_unnamed}")
print(f"Dimensiones actualizadas: {df_merged.shape[0]:,} filas y {df_merged.shape[1]} columnas")

### 6: Inspección General de la Calidad de los Datos
Analizamos los tipos de datos actuales y registros desconocidos.

In [ ]:
# Información general sobre tipos de datos y nulos de Pandas
print("--- INFORMACIÓN GENERAL DE LOS DATOS ---")
df_merged.info()

### 7: Detección de Valores Nulos / Cadenas Vacías
Cuantificamos los nulos que hay en cada columna

In [ ]:
# 1. Calcular el total de nulos por columna
null_counts = df_merged.isnull().sum()

# 2. Calcular el porcentaje de nulos sobre el total de filas (43,000)
null_percentages = (df_merged.isnull().sum() / len(df_merged)) * 100

# 3. Consolidar la información en un DataFrame de resumen
summary_nulls = pd.DataFrame({
    'Valores_Nulos': null_counts,
    'Porcentaje_%': null_percentages.round(2)
})

# 4. Filtrar para mostrar solo las columnas que tienen al menos 1 valor nulo
summary_nulls = summary_nulls[summary_nulls['Valores_Nulos'] > 0].sort_values(by='Valores_Nulos', ascending=False)

print("--- RESUMEN DE COLUMNAS CON VALORES NULOS (NaN) ---")
if not summary_nulls.empty:
    print(summary_nulls)
else:
    print("¡No se encontraron valores nulos (NaN) en ninguna columna!")

### 8: Transformación y Limpieza de Datos
>Realizaremos el formateo y limpieza estandarizada: detectar y corregir errores, manejar datos faltantes y realizar modificaciones adecuadas a las columnas y tipos de datos.
>Agruparemos las variables por bloques temáticos para tener un manejo de datos ordenado

In [ ]:
# --------------------------------------------------------------------------------
# BLOQUE 1: PERFIL DEMOGRÁFICO Y GEOGRÁFICO DEL CLIENTE
# Variables: age, job, marital, education, Kidhome, Teenhome, latitude, longitude
# --------------------------------------------------------------------------------
print("=== BLOQUE 1: PERFIL DEMOGRÁFICO ===")

# 1.1 age (Numérica: imputación por mediana en datos nulos)
df_merged['age'] = pd.to_numeric(df_merged['age'], errors='coerce')
if df_merged['age'].isnull().sum() > 0:
    mediana_edad = df_merged['age'].median()
    df_merged['age'] = df_merged['age'].fillna(mediana_edad)
df_merged['age'] = df_merged['age'].astype(int)

# 1.2 job, marital, education (Categóricas: nulos a 'unknown')
cat_demographic = ['job', 'marital', 'education']
for col in cat_demographic:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].fillna('unknown').astype(str).str.strip()

# 1.3 latitude, longitude no requiere limpieza ni transformar (sin nulos y de tipo float64)

# ----Verificación de Calidad de Datos-----
cols_bloque1 = ['age', 'job', 'marital', 'education', 'Kidhome', 'Teenhome', 'latitude', 'longitude']

verificar_calidad(df_merged, cols_bloque1)

In [ ]:
# ---------------------------------------------------------
# BLOQUE 2: PERFIL FINANCIERO Y CREDITICIO
# Variables: default, housing, loan, Income
# ---------------------------------------------------------
print("=== BLOQUE 2: PERFIL FINANCIERO Y CREDITICIO ===")

# 2.1 Income no requiere limpieza ni transformar (sin nulos y de tipo int64)

# 2.2 default, housing, loan se mapean a 'yes'/'no' y se completan nulos con 'unknown'
fin_cols = ['default', 'housing', 'loan']
mapa_fin = {1.0: 'yes', 0.0: 'no', 1: 'yes', 0: 'no', '1.0': 'yes', '0.0': 'no', 'yes': 'yes', 'no': 'no'}

for col in fin_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].map(mapa_fin).fillna('unknown').astype(str).str.strip().str.lower()

# ----Verificación de Calidad de Datos-----
cols_bloque2 = ['default', 'housing', 'loan', 'Income']

verificar_calidad(df_merged, cols_bloque2)

In [ ]:
# ---------------------------------------------------------
# BLOQUE 3: DATOS DE CONTACTO CON EL CLIENTE
# Variables: contact, duration, date, NumWebVisitsMonth
# ---------------------------------------------------------
print("=== BLOQUE 3: DATOS DE CONTACTO ===")

# 3.1 contact, no requiere limpieza ni transformar (sin nulos y de tipo str)
# 3.2 duration, no requiere limpieza ni transformar (sin nulos y de tipo int64)
# 3.3 NumWebVisitsMonth, no requiere limpieza ni transformar (sin nulos y de tipo int64)

# 3.3 Transformar date (str) a formato de fecha (datetime64)
# Diccionario para mapear meses en español a números
meses_es = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

if 'date' in df_merged.columns:
    # 1. Normalizar texto y reemplazar nombres de meses por números
    date_temp = df_merged['date'].astype(str).str.lower().str.replace('-', '/')
    for mes_nombre, mes_num in meses_es.items():
        date_temp = date_temp.str.replace(mes_nombre, mes_num, regex=False)

    # 2. Convertir a datetime interpretando múltiples formatos
    df_merged['date'] = pd.to_datetime(
        date_temp, 
        format='mixed', 
        dayfirst=True
    )
    
# 3. Eliminar únicamente los nulos reales
    filas_iniciales = len(df_merged)
    df_merged = df_merged.dropna(subset=['date'])
    print(f"✓ Filas eliminadas por nulos en 'date': {filas_iniciales - len(df_merged)}")

    # 4. Extraer mes y año
    df_merged['contact_month'] = df_merged['date'].dt.month.astype(int)
    df_merged['contact_year'] = df_merged['date'].dt.year.astype(int)

# ----Verificación de Calidad de Datos-----
cols_bloque3 = ['contact', 'duration', 'NumWebVisitsMonth', 'date', 'contact_month', 'contact_year']

verificar_calidad(df_merged, cols_bloque3)

In [ ]:
# ---------------------------------------------------------
# BLOQUE 4: PERFIL TEMPORAL Y VARIABLE TARGET
# Variables: y, Dt_Customer
# ---------------------------------------------------------
print("=== BLOQUE 4: PERFIL TEMPORAL Y VARIABLE TARGET ===")

# 4.1 y, no requiere limpieza ni transformar (sin nulos y de tipo str)

# 4.2 --- PROCESAMIENTO DE Dt_Customer ---

# 1. Asegurar formato datetime en Dt_Customer
df_merged['Dt_Customer'] = pd.to_datetime(df_merged['Dt_Customer'], format='mixed', dayfirst=True)

# 2. Extraer año y mes como nuevas columnas
df_merged['Customer_year'] = df_merged['Dt_Customer'].dt.year
df_merged['Customer_month'] = df_merged['Dt_Customer'].dt.month

# 3. Calcular antigüedad en años al momento de la campaña
df_merged['Customer_tenure_year'] = (
    (df_merged['date'] - df_merged['Dt_Customer']).dt.days / 365.25
).astype(int)

# ----Verificación de Calidad de Datos-----
cols_bloque4 = ['y', 'Dt_Customer', 'Customer_year', 'Customer_month', 'Customer_tenure_year']

verificar_calidad(df_merged, cols_bloque4)

In [ ]:
# ---------------------------------------------------------
# BLOQUE 5: HISTORIAL DE CAMPAÑAS DE MARKETING
# Variables: campaign, pdays, previous, poutcome
# ---------------------------------------------------------
print("=== BLOQUE 5: HISTORIAL DE CAMPAÑAS DE MARKETING ===")

# 5.1 campaign, previous, no requiere limpieza ni transformar (sin nulos y de tipo int64)
# 5.2 poutcome, no requiere limpieza ni transformar (sin nulos y de tipo str)

# 5.3 --- PROCESAMIENTO DE pdays ---

# Normalizar nombres de columnas
df_merged.columns = df_merged.columns.str.strip()

# 1. Transformación de 'pdays' (999 -> -1) 
# distinción semántica de que el cliente no fue contactado previamente
if 'pdays' in df_merged.columns:
    df_merged['pdays'] = df_merged['pdays'].replace(999, -1).astype(int)

# 2. Homogeneizar tipos de datos
if 'campaign' in df_merged.columns:
    df_merged['campaign'] = df_merged['campaign'].astype(int)

if 'previous' in df_merged.columns:
    df_merged['previous'] = df_merged['previous'].astype(int)

if 'poutcome' in df_merged.columns:
    df_merged['poutcome'] = df_merged['poutcome'].astype(str).str.strip().str.lower()

# ----Verificación de Calidad de Datos-----
cols_bloque5 = ['campaign', 'pdays', 'previous', 'poutcome']

print("\nVerificación de pdays (post-transformación):")
print(f"Mínimo pdays: {df_merged['pdays'].min()}")
print(f"Máximo pdays: {df_merged['pdays'].max()}")
print(f"Cantidad de clientes sin contacto previo (-1): {(df_merged['pdays'] == -1).sum():,}")

verificar_calidad(df_merged, cols_bloque5)

In [ ]:
# ------------------------------------------------------------------------------
# BLOQUE 6: MACROECONOMÍA Y CLIENTE
# Variables: emp.var.rate, cons.price.idx, cons.conf.idx, euribor3m, nr.employed
# ------------------------------------------------------------------------------
print("=== BLOQUE 6: MACROECONOMÍA Y CLIENTE ===")

# Normalizar nombres de columnas
df_merged.columns = df_merged.columns.str.strip()

macro_cols = ['emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
cols_bloque6 = [col for col in macro_cols if col in df_merged.columns]

# 1. Convertir texto con comas a float (se excluye 'emp.var.rate' que ya es float64)
for col in cols_bloque6:
    if df_merged[col].dtype == 'object' or str(df_merged[col].dtype) == 'str':
        df_merged[col] = df_merged[col].astype(str).str.strip().str.replace(',', '.').astype(float)

# 2. Imputación de nulos en euribor3m y cons.price.idx
cols_imputar = ['euribor3m', 'cons.price.idx']

for col in cols_imputar:
    if col in df_merged.columns and df_merged[col].isnull().sum() > 0:
        # Intento 1: Mediana por mes y año de contacto (mantiene fidelidad histórica)
        if 'contact_year' in df_merged.columns and 'contact_month' in df_merged.columns:
            df_merged[col] = df_merged[col].fillna(
                df_merged.groupby(['contact_year', 'contact_month'])[col].transform('median')
            )
        
        # Intento 2: Mediana global para cualquier nulo remanente
        df_merged[col] = df_merged[col].fillna(df_merged[col].median())

# ----Verificación de Calidad de Datos-----

verificar_calidad(df_merged, cols_bloque6)

In [ ]:
# ---------------------------------------------------------
# AUDITORÍA FINAL Y EXPORTACIÓN DEL DATASET
# ---------------------------------------------------------
print("=== AUDITORÍA FINAL DEL DATASET PROCESADO ===")

# 1. Verificación general de salud del dataset
total_nulos = df_merged.isnull().sum().sum()
filas, columnas = df_merged.shape

print(f"\nDimensiones finales: {filas} filas y {columnas} columnas")
print(f"\nTotal de valores nulos remanentes: {total_nulos}")

if total_nulos > 0:
    print("\n⚠️ Detalle de nulos remanentes por columna:")
    print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])
else:
    print("✅ Excelente: El dataset quedó completamente limpio sin valores nulos.")

# 2. Definir la ruta apuntando a 'data/processed' fuera de 'notebooks'
output_dir = os.path.join('..', 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, 'bank_marketing_cleaned.csv')
df_merged.to_csv(file_path, index=False)

print("-" * 80)
print(f"💾 Dataset exportado exitosamente en: {file_path}")
print("-" * 80)